In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
import optuna

In [3]:
transform_train = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

    ]
)

transform_test = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

In [4]:
train_datasets = datasets.ImageFolder(root='./dataset/train',transform=transform_train)
test_datasets = datasets.ImageFolder(root='./dataset/test',transform=transform_test)

c:\hantorch\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\hantorch\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
from torch.utils.data import random_split
train_data_size = len(train_datasets)

#8:2  train:8   test:2
train_size = int(train_data_size*0.8)
val_size = train_data_size - train_size

print(f'train data size : {train_size}    val data size : {val_size}')

train_dataset , val_dataset = random_split(train_datasets, [train_size,val_size])

#batchsize 최적화를 위해 objective 함수 안에서 처리
#train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True)
#val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=4, shuffle=True)

train data size : 96    val data size : 24


In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = models.resnet34(pretrained=True)
model.fc = nn.Linear(512,3)
model = model.to(device)

In [ ]:

def objective(trial):

    batch_size = trial.suggest_categorical('batch_size',[4,6,8])
    lr = trial.suggest_loguniform('lr',1e-5,1e-3)

    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
    
    optimizer = optim.Adam(model.parameters(), lr = lr)
    criterion = nn.CrossEntropyLoss()
    epochs = 20
    val_loss = 0


    for epoch in range(epochs):
        model.train()
        print(epoch)

        for img,labels in train_dataloader:
            optimizer.zero_grad()
            preds = model(img.to(device))
            loss = criterion(preds,labels.to(device))
            loss.backward()
            optimizer.step()

        
        model.eval()
        with torch.no_grad():
            for img,labels in val_dataloader:
                img = img.to(device)
                labels = labels.to(device)
                preds = model(img)
                val_loss += criterion(preds,labels)


        total_loss = val_loss / len(val_dataloader)

        trial.report(total_loss, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()


    return total_loss
    
    
study = optuna.create_study(direction='minimize')
study.optimize(objective,n_trials=50)



[I 2026-03-06 10:53:28,160] A new study created in memory with name: no-name-441f76ca-a2be-493d-ae8b-0fd2f5abe133
C:\Users\user\AppData\Local\Temp\ipykernel_18292\3224298031.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr',1e-5,1e-3)


0
1
2
3
4
5


In [18]:
print(study.best_trial.params)

{'batch_size': 4, 'lr': 0.00013573022624209873}
